# Agents with Tools vs. Tasks with Tools in CrewAI

Two ways to attach tools in CrewAI, built side by side so the difference is visible:

- **Agent-centric** — the agent owns every tool and decides which to use per task.
- **Task-centric** — each task carries only the tools it needs, narrowing the agent's choices.

Both power the same restaurant FAQ chatbot, backed by a PDF search tool and live web search. The notebook closes by building custom tools with the `@tool` decorator.

## Setup


Libraries used: `crewai` and `crewai-tools` for the agents and tools, `langchain-huggingface` + `sentence-transformers` for local PDF embeddings, and `python-dotenv` for API keys.

In [ ]:
%%capture
%pip install crewai crewai-tools python-dotenv
%pip install langchain-community langchain-huggingface sentence-transformers

In [ ]:
%%capture

from crewai import Agent, Task, Crew, Process
from crewai import LLM
from crewai_tools import PDFSearchTool, SerperDevTool

## Agent-centric vs. task-centric tools

Giving every tool to the agent is flexible but leaves it choosing among all of them on every task. Attaching tools per task constrains the choice, which makes behavior more predictable when a task should only ever use one source.

### The model

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from crewai import LLM

# Load GROQ_API_KEY (and SERPER_API_KEY where needed) from a .env file.
load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError(
        "GROQ_API_KEY is not set. Copy .env.example to .env and add your key "
        "(free at https://console.groq.com/keys)."
    )

# Groq through its OpenAI-compatible endpoint.
llm = LLM(
    model="openai/llama-3.3-70b-versatile",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    max_tokens=2000,
)

### Tools

Two sources: a **PDF search tool** over the restaurant FAQ, and a **web search tool** for anything outside it.

Load API keys and initialize web search.

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv

# Keys come from a .env file - never hard-code them in the notebook.
load_dotenv(find_dotenv(usecwd=True))

for var in ("GROQ_API_KEY", "SERPER_API_KEY"):
    if not os.environ.get(var):
        raise RuntimeError(f"{var} is not set. Add it to your .env file.")

print("API keys loaded.")

In [ ]:
web_search_tool = SerperDevTool()

PDF search over the FAQ, embedded locally with sentence-transformers (no embedding API key needed).

In [ ]:
import warnings
warnings.filterwarnings('ignore') #Keeps Jupyter Notebook clean (not part of functionality)

pdf_search_tool = PDFSearchTool(
    pdf="The-Daily-Dish-FAQ.pdf"   # ships with this project,
    config=dict(
        embedder=dict(
            provider="huggingface",
            config=dict(
                model="sentence-transformers/all-MiniLM-L6-v2"
            )
        )
    )
)

## Approach 1: Agent-centric tools

The agent holds both tools and picks per task.

In [ ]:
agent_centric_agent = Agent(
    role="The Daily Dish Inquiry Specialist",
    goal="""Accurately answer customer questions about The Daily Dish restaurant. 
    You must decide whether to use the restaurant's FAQ PDF or a web search to find the best answer.""",
    backstory="""You are an AI assistant for 'The Daily Dish'.
    You have access to two tools: one for searching the restaurant's FAQ document and another for searching the web.
    Your job is to analyze the user's question and choose the most appropriate tool to find the information needed to provide a helpful response.""",
    tools=[pdf_search_tool, web_search_tool],
    verbose=True,
    allow_delegation=False,
    llm=llm
)

Its task.

In [ ]:
agent_centric_task = Task(
    description="Answer the following customer query: '{customer_query}'. "
                "Analyze the question and use the tools at your disposal (PDF search or web search) to find the most relevant information. "
                "Synthesize the findings into a clear and friendly response.",
    expected_output="A comprehensive and well-formatted answer to the customer's query.",
    agent=agent_centric_agent
)

Assemble the crew.

In [ ]:
agent_centric_crew = Crew(
    agents=[agent_centric_agent],
    tasks=[agent_centric_task],
    process=Process.sequential,
    verbose=False
)

Download the FAQ the PDF tool reads.

In [ ]:
# The FAQ document ships with this project - nothing to download.
import os
assert os.path.exists("The-Daily-Dish-FAQ.pdf"), "The-Daily-Dish-FAQ.pdf is missing"
print("FAQ document ready.")

In [ ]:
def ask(question: str):
    """Send one question to the crew and print the answer."""
    result = agent_centric_crew.kickoff(inputs={"question": question})
    print(result.raw)
    return result


# Try a couple of questions. Call ask("...") with your own to continue.
ask("What are your opening hours?")

With `verbose=True` you can watch which tool the agent chose.

## Approach 2: Task-centric tools

The agent gets no tools; each task supplies the one it needs, so the routing decision moves from the agent to the graph of tasks.

In [ ]:
task_centric_agent = Agent(
    role="Customer Service Specialist",
    goal="Provide exceptional customer service by following a multi-step process to answer customer questions accurately.",
    backstory="""You are an AI assistant for 'The Daily Dish'.
    You are an expert at following instructions. You will be given a sequence of tasks to complete.
    For each task, you will be provided with the specific tool needed to accomplish it.
    Your job is to execute each task diligently and pass the results to the next step.""",
    tools=[], # The agent is not given any tools directly
    verbose=True,
    allow_delegation=False,
    llm=llm
)

Each task carries its own tool.

In [ ]:
faq_search_task = Task(
    description="Search the restaurant's FAQ PDF for information related to the customer's query: '{customer_query}'.",
    expected_output="A snippet of the most relevant information from the PDF, or a statement that the information was not found.",
    tools=[pdf_search_tool], # Tool assigned directly to the task
    agent=task_centric_agent
)

response_drafting_task = Task(
    description="Using the information gathered from the FAQ search, draft a friendly and comprehensive response to the customer's query: '{customer_query}'.",
    expected_output="The final, customer-facing response.",
    agent=task_centric_agent,
    context=[faq_search_task]
)

Assemble the crew.

In [ ]:
task_centric_crew = Crew(
    agents=[task_centric_agent],
    tasks=[faq_search_task, response_drafting_task],
    process=Process.sequential,
    verbose=True
)

In [ ]:
def ask(question: str):
    """Send one question to the crew and print the answer."""
    result = task_centric_crew.kickoff(inputs={"question": question})
    print(result.raw)
    return result


# Try a couple of questions. Call ask("...") with your own to continue.
ask("What are your opening hours?")

## Custom tools

The `@tool` decorator turns any function into something an agent can call.

An `add` tool.

In [ ]:
from crewai.tools import tool
import re

@tool("Add Two Numbers Tool")
def add_numbers(data: str) -> int:
    """
    Extracts and adds integers from the input string.
    Example input: 'add 1 and 2' or '[1,2,3,4]'
    Output: sum of the numbers
    """
    # Find all integers in the string
    numbers = list(map(int, re.findall(r'-?\d+', data)))
    return sum(numbers)

A `multiply` tool.

In [ ]:
from functools import reduce

@tool("Multiply Numbers Tool")
def multiply_numbers(data: str) -> int:
    """
    Extracts and multiplies integers from the input string.
    Example input: 'multiply 2 and 3' or '[2,3,4]'
    Output: the product of all numbers found
    """
    numbers = list(map(int, re.findall(r'-?\d+', data)))
    return reduce(lambda x, y: x * y, numbers, 1)

An agent that extracts the numbers and calls the right tool.

In [ ]:
calculator_agent = Agent(
    role="Calculator",
    goal="Extracts, adds, or multiplies numbers when asked, using the Add Two Numbers and Multiply Numbers tools.",
    backstory="An expert at parsing numeric instructions and computing sums or products.",
    tools=[add_numbers, multiply_numbers],
    llm=llm,
    allow_delegation=False
)

Its task.

In [ ]:
calculation_task = Task(
    description="Extract numbers from '{numbers}' and either add or multiply them, depending on the natural-language instruction.",
    expected_output="An integer result (sum or product) based on the user’s request.",
    agent=calculator_agent
)

The crew.

In [ ]:
crew = Crew(
    agents=[calculator_agent],
    tasks=[calculation_task],
    # verbose=True #Uncomment this to see the steps taken to get the final answer
)

Run it — addition, then multiplication.

In [ ]:
# Inputs for addition…
result = crew.kickoff(inputs={'numbers': 'please add 4, 5, and 6'})
print("Sum result:", result)

In [ ]:
# Inputs for multiplication…
result = crew.kickoff(inputs={'numbers': 'multiply 7 and 8 also 9 dont forget 10'})
print("Product result:", result)

## Author

**Anas AlGhannam**  
[github.com/AnasAlghannam](https://github.com/AnasAlghannam)